In [ ]:

from google.colab import drive
drive.mount('/content/drive')

# Project root
PROJ = "/content/drive/MyDrive/Imbalance"

import os, json, math, random, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

os.makedirs(os.path.join(PROJ, "ckpts"), exist_ok=True)
os.makedirs(os.path.join(PROJ, "metrics"), exist_ok=True)
os.makedirs(os.path.join(PROJ, "figures"), exist_ok=True)
os.makedirs(os.path.join(PROJ, "splits"), exist_ok=True)

%cd $PROJ

SEED = 0
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Mounted at /content/drive
/content/drive/MyDrive/Imbalance
Device: cuda


In [ ]:
CONFIG = {
    "epochs": 50,
    "batch_size": 128,
    "lr": 0.1,
    "num_workers": 2,

    # models to compare
    "backbones": ["resnet18","densenet121"],

    # focal hyperparams to sweep
    "gammas": [1.5, 2.0],
    # alpha: None (no class balance) or "inv_freq" (inverse-frequency weights)
    "alpha_mode": [None, "inv_freq"],


    # options: "baseline" (no rebalancing), "wrs" (WeightedRandomSampler), "ros" (RandomOverSampler)
    "loader_modes": ["baseline","wrs","ros"],

    # optional light augmentations (set True/False to include in runs)
    "use_mixup": False,
    "use_cutmix": False,
    "use_patchmix": False,
    "mixup_alpha": 0.2,
    "cutmix_alpha": 1.0,
    "patch_frac": 0.5,

    # scenarios to include
    "include_lt": True,
    "include_ss": True,
    "include_as": True,
}
print(CONFIG)


{'epochs': 50, 'batch_size': 128, 'lr': 0.1, 'num_workers': 2, 'backbones': ['resnet18', 'densenet121'], 'gammas': [1.5, 2.0], 'alpha_mode': [None, 'inv_freq'], 'loader_modes': ['baseline', 'wrs', 'ros'], 'use_mixup': False, 'use_cutmix': False, 'use_patchmix': False, 'mixup_alpha': 0.2, 'cutmix_alpha': 1.0, 'patch_frac': 0.5, 'include_lt': True, 'include_ss': True, 'include_as': True}


In [ ]:
# Transforms: keep same as your classifier notebooks
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
])

train_data = datasets.CIFAR10("./dataset_cache", train=True, download=True, transform=train_tf)
test_data  = datasets.CIFAR10("./dataset_cache", train=False, download=True, transform=test_tf)
class_labels = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

# Load all three splits
SPLITS = {}
for scn in ["lt","ss","as"]:
    p = os.path.join("splits", f"{scn}.json")
    if os.path.exists(p):
        with open(p) as f:
            data = json.load(f)
        SPLITS[scn] = {
            "indices": data["indices"],
            "counts": {int(k): int(v) for k,v in data["counts"].items()}
        }
        print(f"Loaded split '{scn.upper()}': {len(SPLITS[scn]['indices'])} images")
    else:
        print(f"WARNING: splits/{scn}.json not found — skipping.")
assert len(SPLITS)>0, "No splits found. Run data_splits first."

# Common test loader (always the clean CIFAR-10 test set)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False,
                         num_workers=CONFIG["num_workers"], pin_memory=True)


Loaded split 'LT': 16191 images
Loaded split 'SS': 27500 images
Loaded split 'AS': 9500 images


In [ ]:
def resnet18_cifar(num_classes=10):
    m = models.resnet18(weights=None)
    m.conv1 = nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1,bias=False)
    m.maxpool = nn.Identity()
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

def densenet121_cifar(num_classes=10):
    m = models.densenet121(weights=None)
    m.features.conv0 = nn.Conv2d(3,64,3,1,1,bias=False)
    m.features.pool0 = nn.Identity()
    m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    return m

def make_model(name, num_classes=10):
    if name=="resnet18": return resnet18_cifar(num_classes).to(device)
    if name=="densenet121": return densenet121_cifar(num_classes).to(device)
    raise ValueError("unknown backbone")


In [ ]:
def make_baseline_loader(dataset, indices, batch_size=128):
    subset = Subset(dataset, indices)
    return DataLoader(subset, batch_size=batch_size, shuffle=True,
                      num_workers=CONFIG["num_workers"], pin_memory=True)

def make_wrs_loader(dataset, indices, counts, batch_size=128):
    subset = Subset(dataset, indices)
    # inverse-frequency weights per class
    arr = np.array([counts.get(c,0) for c in range(10)], dtype=np.float64)
    inv = 1.0 / np.maximum(arr, 1.0)
    y_all = np.array(dataset.targets)[np.array(indices)]
    w = inv[y_all]
    sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),
                                    num_samples=len(w), replacement=True)
    return DataLoader(subset, batch_size=batch_size, sampler=sampler,
                      num_workers=CONFIG["num_workers"], pin_memory=True)

def make_ros_loader(dataset, indices, counts, batch_size=128):
    # simple ROS in-memory: upsample each class to max count
    by_class = {c: [] for c in range(10)}
    targets = np.array(dataset.targets)
    for idx in indices:
        by_class[int(targets[idx])].append(idx)
    maxc = max(len(v) for v in by_class.values() if len(v)>0)
    up_idx = []
    rng = np.random.default_rng(SEED)
    for c, idxs in by_class.items():
        if len(idxs)==0: continue
        need = maxc - len(idxs)
        if need>0:
            pick = rng.choice(idxs, size=need, replace=True).tolist()
            up_idx.extend(idxs + pick)
        else:
            up_idx.extend(idxs)
    rng.shuffle(up_idx)
    subset = Subset(dataset, up_idx)
    return DataLoader(subset, batch_size=batch_size, shuffle=True,
                      num_workers=CONFIG["num_workers"], pin_memory=True)


In [ ]:
def rand_bbox(size, lam):
    H, W = size[-2], size[-1]
    cut_rat = math.sqrt(1. - lam)
    cut_w = int(W * cut_rat); cut_h = int(H * cut_rat)
    cx = np.random.randint(W); cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W); y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W); y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2

def apply_mixup(x, y, alpha=0.2):
    if alpha <= 0: return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x2, y2 = x[perm], y[perm]
    x_m = lam*x + (1-lam)*x2
    return x_m, y, y2, lam

def apply_cutmix(x, y, alpha=1.0):
    if alpha <= 0: return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x2, y2 = x[perm], y[perm]
    x1,y1,x2b,y2b = rand_bbox(x.size(), lam)
    x_cut = x.clone()
    x_cut[:,:, y1:y2b, x1:x2b] = x2[:,:, y1:y2b, x1:x2b]
    lam = 1 - ((x2b-x1)*(y2b-y1) / (x.size(-1)*x.size(-2)))  # exact area ratio
    return x_cut, y, y2, lam

def apply_patchmix(x, y, patch_frac=0.5):
    if patch_frac <= 0: return x, y, None, 1.0
    perm = torch.randperm(x.size(0), device=x.device)
    x2, y2 = x[perm], y[perm]
    H, W = x.size(-2), x.size(-1)
    ph, pw = int(H*patch_frac), int(W*patch_frac)
    y1 = np.random.randint(0, H-ph+1); x1 = np.random.randint(0, W-pw+1)
    x_pm = x.clone()
    x_pm[:,:, y1:y1+ph, x1:x1+pw] = x2[:,:, y1:y1+ph, x1:x1+pw]
    lam = 0.5
    return x_pm, y, y2, lam



    Multi-class focal loss with logits.
    gamma > 0 focuses on hard examples.
    alpha:
      - None: no class weighting
      - Tensor of shape [C]: class weights (e.g., inverse frequency)
      - float in (0,1): uniform foreground weight (not typical for multi-class)
    

In [ ]:
class FocalLoss(nn.Module):

    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        if alpha is None:
            self.alpha = None
        else:
            self.alpha = alpha
        self.reduction = reduction

    def forward(self, logits, target):
        # logits: [B,C]; target: [B]
        ce = F.cross_entropy(logits, target, weight=self.alpha, reduction="none")
        pt = torch.softmax(logits, dim=1).gather(1, target.view(-1,1)).squeeze(1).clamp_min(1e-8)
        loss = ((1-pt)**self.gamma) * ce
        if self.reduction=="mean":
            return loss.mean()
        elif self.reduction=="sum":
            return loss.sum()
        return loss


In [ ]:
def make_alpha_vector(counts_dict, mode=None, device=device):
    # mode: None or "inv_freq"
    if mode is None:
        return None
    arr = np.array([counts_dict.get(c, 0) for c in range(10)], dtype=np.float64)
    arr = np.maximum(arr, 1.0)
    inv = 1.0 / arr
    inv = inv / inv.mean()  # normalize around 1
    return torch.tensor(inv, dtype=torch.float32, device=device)


In [ ]:
def train_one_epoch(model, loader, criterion, opt,
                    use_mixup=False, use_cutmix=False, use_patchmix=False,
                    mixup_alpha=0.2, cutmix_alpha=1.0, patch_frac=0.5):
    model.train()
    tot_loss, corr, n = 0.0, 0, 0
    for x,y in loader:
        x,y = x.to(device), y.to(device)

        lam, y2 = 1.0, None
        if use_cutmix:
            x, y, y2, lam = apply_cutmix(x,y,alpha=cutmix_alpha)
        elif use_mixup:
            x, y, y2, lam = apply_mixup(x,y,alpha=mixup_alpha)
        elif use_patchmix:
            x, y, y2, lam = apply_patchmix(x,y,patch_frac=patch_frac)

        logits = model(x)
        if y2 is None:
            loss = criterion(logits, y)
        else:
            loss = lam*criterion(logits, y) + (1-lam)*criterion(logits, y2)

        opt.zero_grad(set_to_none=True)
        loss.backward(); opt.step()

        pred = logits.argmax(1)
        corr += (pred==y).sum().item()
        tot_loss += loss.item() * x.size(0)
        n += x.size(0)
    return tot_loss/n, corr/n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    tot_loss, corr, n = 0.0, 0, 0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        pred = logits.argmax(1)
        corr += (pred==y).sum().item()
        tot_loss += loss.item() * x.size(0)
        n += x.size(0)
    return tot_loss/n, corr/n

@torch.no_grad()
def get_preds(model, loader):
    model.eval()
    ys, ps = [], []
    for x,y in loader:
        x = x.to(device)
        pred = model(x).argmax(1).cpu().numpy()
        ys.append(y.numpy()); ps.append(pred)
    return np.concatenate(ys), np.concatenate(ps)

def confusion_matrix(y_true, y_pred, K=10):
    cm = np.zeros((K,K), dtype=np.int64)
    for t,p in zip(y_true, y_pred): cm[t,p]+=1
    return cm

def macro_f1(cm):
    prec = np.diag(cm) / np.clip(cm.sum(0),1,None)
    rec  = np.diag(cm) / np.clip(cm.sum(1),1,None)
    f1   = 2*prec*rec / np.clip(prec+rec,1e-12,None)
    return float(np.nanmean(f1))

def balanced_acc(cm):
    rec  = np.diag(cm) / np.clip(cm.sum(1),1,None)
    return float(np.nanmean(rec))


In [ ]:
from sklearn.metrics import classification_report, precision_recall_fscore_support
import pandas as pd

def make_loader(mode, dataset, indices, counts, bs):
    if mode=="baseline": return make_baseline_loader(dataset, indices, bs)
    if mode=="wrs":      return make_wrs_loader(dataset, indices, counts, bs)
    if mode=="ros":      return make_ros_loader(dataset, indices, counts, bs)
    raise ValueError("loader mode invalid")

summary_rows = []

for scn_key, split in SPLITS.items():
    if (scn_key=="lt" and not CONFIG["include_lt"]) or \
       (scn_key=="ss" and not CONFIG["include_ss"]) or \
       (scn_key=="as" and not CONFIG["include_as"]):
        continue

    indices = split["indices"]
    counts  = split["counts"]

    for loader_mode in CONFIG["loader_modes"]:
        # (Optional) you may choose to skip ROS/WRS for AS if you prefer
        print(f"\n============== Scenario {scn_key.upper()} | loader={loader_mode} ==============")
        train_loader = make_loader(loader_mode, train_data, indices, counts, CONFIG["batch_size"])

        for backbone in CONFIG["backbones"]:
            for gamma in CONFIG["gammas"]:
                for alpha_mode in CONFIG["alpha_mode"]:
                    tag = f"focal_g{gamma}_{alpha_mode or 'none'}_{loader_mode}"

                    # Build alpha vector if requested
                    alpha_vec = make_alpha_vector(counts, mode=alpha_mode)

                    model = make_model(backbone, 10).to(device)
                    criterion = FocalLoss(gamma=gamma, alpha=alpha_vec, reduction="mean")

                    opt   = torch.optim.SGD(model.parameters(), lr=CONFIG["lr"], momentum=0.9, weight_decay=5e-4)
                    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG["epochs"])

                    best_val=-1.0; best_state=None
                    history={"train_loss":[],"val_loss":[],"train_acc":[],"val_acc":[]}

                    for ep in range(1, CONFIG["epochs"]+1):
                        tl, ta = train_one_epoch(
                            model, train_loader, criterion, opt,
                            use_mixup=CONFIG["use_mixup"],
                            use_cutmix=CONFIG["use_cutmix"],
                            use_patchmix=CONFIG["use_patchmix"],
                            mixup_alpha=CONFIG["mixup_alpha"],
                            cutmix_alpha=CONFIG["cutmix_alpha"],
                            patch_frac=CONFIG["patch_frac"]
                        )
                        vl, va = evaluate(model, test_loader, criterion)
                        sched.step()

                        history["train_loss"].append(tl); history["train_acc"].append(ta)
                        history["val_loss"].append(vl);   history["val_acc"].append(va)

                        if va > best_val:
                            best_val = va
                            best_state = {k:v.cpu() for k,v in model.state_dict().items()}

                        if ep==1 or ep%5==0:
                            print(f"{scn_key.upper()} | {backbone} | {tag} | Ep {ep:02d} | "
                                  f"train_acc={ta*100:.1f}%  val_acc={va*100:.1f}%")

                    # restore best & compute headline metrics on test
                    if best_state: model.load_state_dict(best_state)
                    y_true, y_pred = get_preds(model, test_loader)
                    cm   = confusion_matrix(y_true, y_pred, K=10)
                    acc  = (y_true==y_pred).mean()
                    mF1  = macro_f1(cm)
                    bAcc = balanced_acc(cm)
                    print(f"TEST — Acc {acc*100:.2f}% | Macro-F1 {mF1*100:.2f}% | Balanced Acc {bAcc*100:.2f}%")

                    # save checkpoint + metrics
                    ckpt_path = os.path.join("ckpts", f"{backbone}_{scn_key.upper()}_{tag}.pt")
                    torch.save(model.state_dict(), ckpt_path)

                    met_path = os.path.join("metrics", f"{backbone}_{scn_key.upper()}_{tag}.json")
                    with open(met_path,"w") as f:
                        json.dump({
                            "scenario": scn_key.upper(),
                            "backbone": backbone,
                            "loader": loader_mode,
                            "gamma": gamma,
                            "alpha_mode": alpha_mode,
                            "epochs": CONFIG["epochs"],
                            "test_acc": float(acc),
                            "macro_f1": float(mF1),
                            "balanced_acc": float(bAcc),
                            "history": history
                        }, f, indent=2)

                    # classification report
                    rep_txt = classification_report(y_true, y_pred, target_names=class_labels, digits=2)
                    rep_path = os.path.join("metrics", f"{backbone}_{scn_key.upper()}_{tag}_report.txt")
                    with open(rep_path, "w") as f: f.write(rep_txt)
                    print("Saved report:", rep_path)

                    # confusion matrix fig
                    plt.figure(figsize=(6.2,5.4))
                    plt.imshow(cm, cmap="Blues", interpolation='nearest')
                    plt.title(f"Confusion Matrix — {scn_key.upper()} {backbone} ({tag})")
                    plt.colorbar()
                    ticks = np.arange(len(class_labels))
                    plt.xticks(ticks, class_labels, rotation=30, ha='right')
                    plt.yticks(ticks, class_labels)
                    plt.xlabel('Predicted'); plt.ylabel('True')
                    plt.tight_layout()
                    fig_cm = os.path.join("figures", f"cm_{backbone}_{scn_key.upper()}_{tag}.png")
                    plt.savefig(fig_cm, dpi=180); plt.show()

                    # per-class bars
                    prec, rec, f1, sup = precision_recall_fscore_support(
                        y_true, y_pred, labels=range(10), zero_division=0)
                    x = np.arange(len(class_labels)); w = 0.28
                    plt.figure(figsize=(11,4))
                    plt.bar(x-w, prec, width=w, label='Precision')
                    plt.bar(x,   rec,  width=w, label='Recall')
                    plt.bar(x+w, f1,   width=w, label='F1')
                    plt.ylim(0,1.0); plt.legend()
                    plt.xticks(x, class_labels, rotation=30, ha='right')
                    plt.ylabel("Score"); plt.title(f"Per-class — {scn_key.upper()} {backbone} ({tag})")
                    plt.tight_layout()
                    fig_bar = os.path.join("figures", f"perclass_{backbone}_{scn_key.upper()}_{tag}.png")
                    plt.savefig(fig_bar, dpi=180); plt.show()

                    # collect for summary
                    summary_rows.append([scn_key.upper(), backbone, loader_mode,
                                         gamma, alpha_mode or "none",
                                         round(mF1*100,2), round(bAcc*100,2), round(acc*100,2)])


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
cols = ["Scenario","Model","Loader","Gamma","AlphaMode","Macro-F1 (%)","Balanced Acc (%)","Overall Acc (%)"]
df = pd.DataFrame(summary_rows, columns=cols).sort_values(["Scenario","Model","Loader","Gamma","AlphaMode"])
display(df)
out_csv = os.path.join("metrics", "summary_focal.csv")
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)


,Scenario,Model,Loader,Gamma,AlphaMode,Macro-F1 (%),Balanced Acc (%),Overall Acc (%)
53,AS,densenet121,baseline,1.5,inv_freq,74.33,74.23,74.23
52,AS,densenet121,baseline,1.5,none,70.62,70.30,70.30
55,AS,densenet121,baseline,2.0,inv_freq,71.57,71.38,71.38
54,AS,densenet121,baseline,2.0,none,70.39,70.13,70.13
68,AS,densenet121,ros,1.5,none,82.44,82.32,82.32
...,...,...,...,...,...,...,...,...
42,SS,resnet18,ros,2.0,none,83.93,83.94,83.94
33,SS,resnet18,wrs,1.5,inv_freq,81.72,81.49,81.49
32,SS,resnet18,wrs,1.5,none,82.75,82.73,82.73
35,SS,resnet18,wrs,2.0,inv_freq,80.96,80.90,80.90


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-2292135841.py", line 5, in <cell line: 0>
    df.to_csv(out_csv, index=False)
  File "/usr/local/lib/python3.12/dist-packages/pandas/util/_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/generic.py", line 3967, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py", line 1014, in to_csv
    csv_formatter.save()
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/formats/csvs.py", line 251, in save
    with get_handle(
         ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/common.py", line 719, in get